convert_to_rh.ipynb

Convert Pencil Code snapshot into RH-compatible HDF5 atmosphere file.

Author: Sanket Wavhal

In [1]:
# ------------------------------------------------------------
# IMPORT LIBRARIES
# ------------------------------------------------------------
import pencil as pc
import numpy as np
import h5py
import matplotlib.pyplot as plt

In [ ]:
# ------------------------------------------------------------
# MULTI-SNAPSHOT RH PIPELINE
# ------------------------------------------------------------

import os

# ----------------------------
# SETTINGS
# ----------------------------
datadir = "../data"

# 👉 OPTION 1: manual list
# snapshots = [109, 110, 111, 112]

# 👉 OPTION 2 (AUTO): uncomment if needed
snapshots = sorted([int(f.replace("VAR", "")) 
                    for f in os.listdir(datadir) if "VAR" in f])

output_dir = "../output"
os.makedirs(output_dir, exist_ok=True)

# Toggle plotting (VERY IMPORTANT for speed)
PLOT = False


# ------------------------------------------------------------
# FUNCTION TO PROCESS ONE SNAPSHOT
# ------------------------------------------------------------
def process_snapshot(ivar):

    print(f"\n🚀 Processing snapshot {ivar}")

    # ----------------------------
    # READ PARAMETERS
    # ----------------------------
    param = pc.read.param(datadir=datadir)

    # ----------------------------
    # READ VARIABLES
    # ----------------------------
    var = pc.read.varraw(
        ivar=ivar,
        var_list=['uu', 'lnrho', 'lnTT', 'yH'],
        datadir=datadir,
        trimall=True
    )

    # ----------------------------
    # CONVERT VARIABLES
    # ----------------------------
    T   = np.exp(var.lnTT) * param.unit_temperature         # K
    rho = np.exp(var.lnrho) * param.unit_density * 1000     # kg/m^3
    vz  = var.uz * param.unit_velocity * 1e-2               # m/s
    yH  = var.yH

    nx, ny, nz = T.shape
    print(f"Initial grid: nx={nx}, ny={ny}, nz={nz}")

    # ----------------------------
    # RECONSTRUCT Z (NO GRID FILE)
    # ----------------------------
    z = np.linspace(param.xyz0[2], param.xyz1[2], nz)
    z = z * param.unit_length * 1e-2                        # cm → m

    # ----------------------------
    # HEIGHT FILTER
    # ----------------------------
    z_min, z_max = -0.3e6, 15e6
    mask = (z >= z_min) & (z <= z_max)

    z   = z[mask]
    T   = T[:, :, mask]
    rho = rho[:, :, mask]
    vz  = vz[:, :, mask]
    yH  = yH[:, :, mask]

    nx, ny, nz = T.shape
    print(f"After height cut: nx={nx}, ny={ny}, nz={nz}")
    print(f"z range: {z.min():.2e} → {z.max():.2e} m")

    # ----------------------------
    # PHYSICS
    # ----------------------------
    xH  = 1.0
    xHe = 0.081
    mp  = 1.6726219e-27

    mu0 = 1 + 4 * xHe
    mu  = mu0 / (1 + yH * xH)

    n_total  = rho / (mu * mp)
    ne       = yH * n_total

    # ----------------------------
    # REVERSE Z (RH REQUIREMENT)
    # ----------------------------
    T  = T[:, :, ::-1]
    vz = vz[:, :, ::-1]
    ne = ne[:, :, ::-1]
    z  = z[::-1]

    # ----------------------------
    # HYDROGEN POPULATION
    # ----------------------------
    nhydr = n_total[:, :, ::-1]                         # reverse z
    nhydr = nhydr[np.newaxis, np.newaxis, ...]          # (nt, nhydr, nx, ny, nz)

    # ----------------------------
    # ADD TIME DIMENSION
    # ----------------------------
    T  = T[np.newaxis, ...]
    vz = vz[np.newaxis, ...]
    ne = ne[np.newaxis, ...]
    z  = z[np.newaxis, :]

    nt, nx, ny, nz = T.shape
    print(f"RH grid: nt={nt}, nx={nx}, ny={ny}, nz={nz}")

    # ----------------------------
    # OPTIONAL VALIDATION PLOTS
    # ----------------------------
    if PLOT:
        ix, iy = nx//2, ny//2
        plt.figure()
        plt.plot(T[0, ix, iy, :], z[0, :])
        plt.title(f"T snapshot {ivar}")
        plt.show()

    # ----------------------------
    # WRITE FILE
    # ----------------------------
    filename = os.path.join(output_dir, f"rh_atmosphere_{ivar}.hdf5")

    with h5py.File(filename, "w") as f:

        # Coordinates
        f.create_dataset("z", data=z)

        x = np.linspace(param.xyz0[0], param.xyz1[0], nx) * param.unit_length * 1e-2
        y = np.linspace(param.xyz0[1], param.xyz1[1], ny) * param.unit_length * 1e-2
        f.create_dataset("x", data=x)
        f.create_dataset("y", data=y)

        # Variables
        f.create_dataset("temperature", data=T)
        f.create_dataset("velocity_z", data=vz)
        f.create_dataset("electron_density", data=ne)
        f.create_dataset("hydrogen_populations", data=nhydr)

        # Snapshot index
        f.create_dataset("snapshot_number", data=[ivar])

        # Attributes
        f.attrs["nt"] = nt
        f.attrs["nx"] = nx
        f.attrs["ny"] = ny
        f.attrs["nz"] = nz
        f.attrs["has_B"] = 0

    print(f"✅ Saved: {filename}")

In [11]:
# ------------------------------------------------------------
# RUN PIPELINE FOR ALL SNAPSHOTS
# ------------------------------------------------------------
for ivar in snapshots:
    process_snapshot(ivar)

print("\n🎉 ALL SNAPSHOTS PROCESSED!")


🚀 Processing snapshot 109

The file VAR109 contains: ux, uy, uz, lnrho, lnTT, ax, ay, az, qx, qy, qz, shock, yH, Qrad, kapparho, hypvisx, hypvisy, hypvisz, hypresx, hypresy, hypresz
Will read only: ux, uy, uz, lnrho, lnTT, yH

The grid dimension is 198, 198, 726

 t = 79.75000312192645

Initial grid: nx=192, ny=192, nz=720
After height cut: nx=192, ny=192, nz=319
z range: -2.83e+05 → 1.50e+07 m
RH grid: nt=1, nx=192, ny=192, nz=319
✅ Saved: ../outputs\rh_atmosphere_109.hdf5

🚀 Processing snapshot 110

The file VAR110 contains: ux, uy, uz, lnrho, lnTT, ax, ay, az, qx, qy, qz, shock, yH, Qrad, kapparho, hypvisx, hypvisy, hypvisz, hypresx, hypresy, hypresz
Will read only: ux, uy, uz, lnrho, lnTT, yH

The grid dimension is 198, 198, 726

 t = 80.0000098714855

Initial grid: nx=192, ny=192, nz=720
After height cut: nx=192, ny=192, nz=319
z range: -2.83e+05 → 1.50e+07 m
RH grid: nt=1, nx=192, ny=192, nz=319
✅ Saved: ../outputs\rh_atmosphere_110.hdf5

🎉 ALL SNAPSHOTS PROCESSED!


In [ ]:
# ============================================================
# RH 3D ATMOSPHERE FROM PENCIL (MODULAR + TOGGLE FRIENDLY)
# ============================================================

import pencil as pc
import numpy as np
import matplotlib.pyplot as plt
import xdrlib3 as xdrlib
import os

# ============================================================
# SETTINGS
# ============================================================

datadir = "data"
ivar = 109
output_file = "rh_3d.atmos"

# ============================================================
# READ DATA
# ============================================================

param = pc.read.param(datadir=datadir)

var = pc.read.varraw(
    ivar=ivar,
    var_list=['uu', 'lnrho', 'lnTT', 'yH'],
    datadir=datadir,
    trimall=True
)

print("\n✅ Data loaded")

# ============================================================
# CONVERT TO PHYSICAL UNITS
# ============================================================

T   = np.exp(var.lnTT) * param.unit_temperature
rho = np.exp(var.lnrho) * param.unit_density * 1000

vx  = var.ux * param.unit_velocity * 1e-2
vy  = var.uy * param.unit_velocity * 1e-2
vz  = var.uz * param.unit_velocity * 1e-2

yH  = var.yH

nx, ny, nz = T.shape
print(f"Initial grid: {nx} x {ny} x {nz}")

# ============================================================
# HEIGHT GRID
# ============================================================

z = np.linspace(param.xyz0[2], param.xyz1[2], nz)
z = z * param.unit_length * 1e-2   # m

# ============================================================
# OPTIONAL: HEIGHT FILTER (toggle ON/OFF)
# ============================================================

if True:
    z_min = -0.3e6
    z_max = 15e6

    mask = (z >= z_min) & (z <= z_max)

    z   = z[mask]
    T   = T[:, :, mask]
    rho = rho[:, :, mask]
    vx  = vx[:, :, mask]
    vy  = vy[:, :, mask]
    vz  = vz[:, :, mask]
    yH  = yH[:, :, mask]

    print(f"After height cut: {T.shape}")

# ============================================================
# OPTIONAL: SUBCUBE SELECTION (toggle ON/OFF)
# ============================================================

if True:
    X_START, X_END, X_STEP = 80, 112, 1
    Y_START, Y_END, Y_STEP = 80, 112, 1

    T   = T[X_START:X_END:X_STEP, Y_START:Y_END:Y_STEP, :]
    rho = rho[X_START:X_END:X_STEP, Y_START:Y_END:Y_STEP, :]
    vx  = vx[X_START:X_END:X_STEP, Y_START:Y_END:Y_STEP, :]
    vy  = vy[X_START:X_END:X_STEP, Y_START:Y_END:Y_STEP, :]
    vz  = vz[X_START:X_END:X_STEP, Y_START:Y_END:Y_STEP, :]
    yH  = yH[X_START:X_END:X_STEP, Y_START:Y_END:Y_STEP, :]

    print(f"Subcube selected: {T.shape}")

# ============================================================
# DENSITIES
# ============================================================

xH  = 1.0
xHe = 0.081
mp  = 1.6726219e-27

mu0 = 1 + 4 * xHe
mu  = mu0 / (1 + yH * xH)

n_total = rho / (mu * mp)
ne      = yH * n_total

# ----------------------------
# HYDROGEN POPULATIONS (2 LEVEL)
# ----------------------------

nH0 = n_total * (1 - yH)
nHp = n_total * yH

# ============================================================
# Z-FLIP (REQUIRED BY RH)
# ============================================================

T   = T[:, :, ::-1]
vx  = vx[:, :, ::-1]
vy  = vy[:, :, ::-1]
vz  = vz[:, :, ::-1]
ne  = ne[:, :, ::-1]
nH0 = nH0[:, :, ::-1]
nHp = nHp[:, :, ::-1]
z   = z[::-1]

nx, ny, nz = T.shape
print(f"Final grid: {nx} x {ny} x {nz}")

# ============================================================
# OPTIONAL: VERIFICATION PLOTS (toggle ON/OFF)
# ============================================================

if False:
    mid = nz // 2
    plt.figure()
    plt.imshow(T[:, :, mid].T, origin='lower')
    plt.colorbar()
    plt.title("Temperature slice")

    plt.figure()
    plt.imshow(np.log10(ne[:, :, mid]).T, origin='lower')
    plt.colorbar()
    plt.title("log ne")

    plt.show()

# ============================================================
# WRITE RH 3D ATMOS (XDR)
# ============================================================

def write_rh_3d_xdr(filename):

    print(f"\n📝 Writing: {filename}")

    # ---- unit conversion ----
    z_km  = z / 1e3
    vx_km = vx / 1e3
    vy_km = vy / 1e3
    vz_km = vz / 1e3

    dx = (param.xyz1[0] - param.xyz0[0]) * param.unit_length * 1e-2 / (nx - 1)
    dy = (param.xyz1[1] - param.xyz0[1]) * param.unit_length * 1e-2 / (ny - 1)

    dx_km = dx / 1e3
    dy_km = dy / 1e3

    vturb = np.zeros_like(T)

    NHydr = 2
    TOP = 1
    BOTTOM = 2

    p = xdrlib.Packer()

    # HEADER
    p.pack_int(nx)
    p.pack_int(ny)
    p.pack_int(nz)

    p.pack_int(NHydr)

    p.pack_int(TOP)
    p.pack_int(BOTTOM)

    p.pack_double(dx_km)
    p.pack_double(dy_km)

    for val in z_km:
        p.pack_double(float(val))

    # helper
    def pack_3d(arr):
        for v in np.ascontiguousarray(arr).ravel(order='C'):
            p.pack_double(float(v))

    # DATA
    pack_3d(T)
    pack_3d(ne)
    pack_3d(vturb)
    pack_3d(vx_km)
    pack_3d(vy_km)
    pack_3d(vz_km)

    pack_3d(nH0)
    pack_3d(nHp)

    # write
    with open(filename, "wb") as f:
        f.write(p.get_buffer())

    print("✅ File written")

# ============================================================
# RUN
# ============================================================

write_rh_3d_xdr(output_file)

size = os.path.getsize(output_file) / 1e6
print(f"\n📦 File size: {size:.2f} MB")

print("\n🎉 READY FOR RH 3D")